# Dask vs Non‑Dask STAC Processing — Extended Notebook

This notebook demonstrates: 

- Processing **Landsat C2 L2** (example) **and** **Sentinel-2** items from a STAC API
- Comparison of **without Dask** vs **with Dask** (chunked) workflows
- **Benchmarking**: timings and memory (using `time` and `psutil`)
- **Dask LocalCluster** setup and how to access the dashboard

Run cells sequentially. If you are in a fresh environment, uncomment the install cell. The actual downloads depend on STAC items availability and internet access.

Created for you by ChatGPT — includes code blocks you can run as-is.


In [ ]:
# Install dependencies if needed (uncomment to run in a fresh kernel)
# !pip install pystac-client rioxarray xarray dask[complete] distributed psutil matplotlib nbformat

## Imports and helper functions

In [ ]:
import time
import gc
import os
import psutil
import xarray as xr
import rioxarray
from pystac_client import Client
from distributed import Client as DaskClient, LocalCluster

def memory_rss_mb():
    proc = psutil.Process(os.getpid())
    return proc.memory_info().rss / (1024 * 1024)

def fetch_one_item(collection='landsat-c2-l2', bbox=[77.5, 23.0, 78.0, 23.5]):
    catalog = Client.open('https://earth-search.aws.element84.com/v1')
    search = catalog.search(collections=[collection], bbox=bbox, limit=1)
    items = list(search.get_items())
    if len(items) == 0:
        raise RuntimeError(f'No items found for collection={collection} and bbox={bbox}')
    return items[0]

print('psutil version available, current RSS (MB):', memory_rss_mb())

## 1) Get a Landsat item (single) from STAC and get Red/NIR URLs

In [ ]:
landsat_item = fetch_one_item(collection='landsat-c2-l2')
landsat_item


In [ ]:
# Inspect assets and pick common band keys; assets differ by provider/item
print(list(landsat_item.assets.keys()))

# Common band keys for Landsat C2 L2 may be 'red' and 'nir08' (but verify per-item)
red_url = landsat_item.assets.get('red').href if landsat_item.assets.get('red') else None
nir_url = landsat_item.assets.get('nir08').href if landsat_item.assets.get('nir08') else None
red_url, nir_url

### 1.1 NDVI — Landsat — Without Dask (eager load)

In [ ]:
gc.collect()
start_mem = memory_rss_mb()
start = time.perf_counter()

# NOTE: these calls will load data into memory (eager) via rasterio
red = rioxarray.open_rasterio(red_url).squeeze()   # remove band dim if present
nir = rioxarray.open_rasterio(nir_url).squeeze()

ndvi_no_dask = (nir - red) / (nir + red)
mean_no_dask = float(ndvi_no_dask.mean().item())

end = time.perf_counter()
end_mem = memory_rss_mb()

print(f"Time (s) - no Dask: {end - start:.2f}")
print(f"Memory RSS (MB) before: {start_mem:.1f}  after: {end_mem:.1f}")
print('NDVI mean (no Dask):', mean_no_dask)

### 1.2 NDVI — Landsat — With Dask (chunked, lazy)

In [ ]:
gc.collect()
start_mem = memory_rss_mb()
start = time.perf_counter()

# Open with chunking to enable Dask (lazy)
red_d = rioxarray.open_rasterio(red_url, chunks={'x': 2048, 'y': 2048}).squeeze()
nir_d = rioxarray.open_rasterio(nir_url, chunks={'x': 2048, 'y': 2048}).squeeze()

ndvi_dask = (nir_d - red_d) / (nir_d + red_d)
# computation happens on .compute(); measure time and memory during compute
mid = time.perf_counter()
mean_dask = float(ndvi_dask.mean().compute())

end = time.perf_counter()
end_mem = memory_rss_mb()

print(f"Time (s) - open+compute with Dask: {end - start:.2f} (open->{mid-start:.2f}, compute->{end-mid:.2f})")
print(f"Memory RSS (MB) before: {start_mem:.1f}  after: {end_mem:.1f}")
print('NDVI mean (Dask):', mean_dask)

## 2) Sentinel-2 example from STAC (single item)

In [ ]:
sentinel_item = fetch_one_item(collection='sentinel-s2-l2a')
sentinel_item

In [ ]:
print(list(sentinel_item.assets.keys()))

# Sentinel-2 band names vary; common ones: B04 (red), B08 (nir)
red_s2 = sentinel_item.assets.get('B04').href if sentinel_item.assets.get('B04') else None
nir_s2 = sentinel_item.assets.get('B08').href if sentinel_item.assets.get('B08') else None
red_s2, nir_s2

### 2.1 NDVI — Sentinel-2 — Without Dask (eager)

In [ ]:
gc.collect()
start_mem = memory_rss_mb()
start = time.perf_counter()

red_s = rioxarray.open_rasterio(red_s2).squeeze()
nir_s = rioxarray.open_rasterio(nir_s2).squeeze()

ndvi_s_no = (nir_s - red_s) / (nir_s + red_s)
mean_s_no = float(ndvi_s_no.mean().item())

end = time.perf_counter()
end_mem = memory_rss_mb()
print(f"Time (s) - Sentinel no Dask: {end-start:.2f}")
print(f"Memory RSS (MB) before: {start_mem:.1f}  after: {end_mem:.1f}")
print('Sentinel NDVI mean (no Dask):', mean_s_no)

### 2.2 NDVI — Sentinel-2 — With Dask (chunked)

In [ ]:
gc.collect()
start_mem = memory_rss_mb()
start = time.perf_counter()

red_s_d = rioxarray.open_rasterio(red_s2, chunks={'x': 2048, 'y': 2048}).squeeze()
nir_s_d = rioxarray.open_rasterio(nir_s2, chunks={'x': 2048, 'y': 2048}).squeeze()

ndvi_s_dask = (nir_s_d - red_s_d) / (nir_s_d + red_s_d)
mid = time.perf_counter()
mean_s_dask = float(ndvi_s_dask.mean().compute())

end = time.perf_counter()
end_mem = memory_rss_mb()

print(f"Time (s) - open+compute with Dask (Sentinel): {end - start:.2f} (open->{mid-start:.2f}, compute->{end-mid:.2f})")
print(f"Memory RSS (MB) before: {start_mem:.1f}  after: {end_mem:.1f}")
print('Sentinel NDVI mean (Dask):', mean_s_dask)

## 3) Benchmark summary
The cells above print timing and memory numbers for each stage. Use these results to compare the behavior of eager vs lazy (Dask) workflows.

## 4) Dask LocalCluster & Dashboard
Start a Dask LocalCluster to take advantage of parallel compute. When you create a `LocalCluster`, it prints a `dashboard_link` you can open in your browser.

In [ ]:
# Start a LocalCluster and Client (adjust n_workers and threads_per_worker to match your machine)
cluster = LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='4GB')
client = DaskClient(cluster)
print(client)
print('\nDashboard link:', client.dashboard_link)

# Example: re-run a small compute using the cluster for Sentinel NDVI (uses the previously opened chunked datasets)
# If the dask-backed objects were created before the client, they will still use the cluster when compute() is called.
start = time.perf_counter()
mean_s_dask_cluster = float(ndvi_s_dask.mean().compute())
end = time.perf_counter()
print(f'Recomputed Sentinel NDVI mean using cluster in {end - start:.2f}s (value: {mean_s_dask_cluster})')

# Cleanup when done
# client.close(); cluster.close()   # uncomment to close when you're finished

## Notes, tips and caveats

- The timings and memory measurements are *approximate*; network speed, STAC server responsiveness, and local disk cache will change results.
- `rioxarray.open_rasterio(..., chunks=...)` enables dask-backed xarray objects. Avoid calling `.values` or `.load()` if you want to stay lazy.
- Use `client.dashboard_link` to open the Dask dashboard: it helps inspect task graphs, memory, and worker activity.
- For robust benchmarking, run multiple trials and consider warming caches (or disabling caching) depending on what you're measuring.
